In [2]:
import glob
import pickle
import pandas as pd
from ete3 import Tree
import seaborn as sns
import matplotlib.pyplot as plt
import datetime
import os

# Get the current date in the desired format
current_date = datetime.datetime.now().strftime("%Y-%m-%d")

# Fraction of non-MP edges

In [10]:
# Fraction of non-MP edges
all_tree_list = []
for file in glob.glob("*.p"):
    file = os.path.realpath(file)
    with open(file, "rb") as f:
        data = pickle.load(f)
    data_name = file.split("/")[-1][:-2]
    print(data_name)
    this_tree_list = []
    for tree in data:
        num_leaves = len(tree)
        num_non_mp_edges = sum(data[tree])
        frac_non_mp_edges = sum(data[tree]) / (len(tree)-1)
        this_tree_list.append([data_name, num_leaves, num_non_mp_edges, frac_non_mp_edges])
    all_tree_list += this_tree_list

4leaf4site_test
30leaf_perfect_distinct_trees_train
harrington_tiny_test
larch_harrington-small_2024-06-10
30leaf_perfect_distinct_trees_small
30leaf_perfect_distinct_trees_test
10leaf_perfect
4leaf_train
10leaf_perfect_distinct_trees_train
10leaf_perfect_small
larch_harrington_2024-06-05
harrington-small_2024-06-10_train
10leaf_perfect_distinct_trees_test
30leaf_perfect
4leaf4site_train
4leaf
harrington-small_0_to_50_taxa_subset
harrington-small_2024-06-10_test
30leaf_perfect_distinct_trees
4leaf4site
4leaf_test
tiny_test
harrington-small_0_to_30_sites_subset


In [13]:
df = pd.DataFrame(all_tree_list, columns = ["data_name", "num_leaves", "num_non_mp_edges", "frac_non_mp_edges"])
df = df.sort_values("data_name")
df.to_csv(f"all_data_stats_{current_date}.csv")

In [14]:
plt.clf()
plt.figure(figsize=(10, 10))
sns.boxplot(data=df, x="frac_non_mp_edges", y="data_name")
plt.tight_layout()
plt.savefig("fractions_non_mp_edges_boxplot.pdf")

# Number of Mutations per Edge

In [2]:
def hamming_distance(seq1, seq2):
    # Check if the lengths of the sequences are equal
    if len(seq1) != len(seq2):
        raise ValueError("Sequences must be of equal length to compute Hamming distance.")
    
    # Count the number of positions where elements are different
    distance = sum(el1 != el2 for el1, el2 in zip(seq1, seq2))
    
    return distance

In [16]:
# Number of mutations per edge
restricted_files=["30leaf_perfect_distinct_trees_train", "10leaf_perfect_distinct_trees_train", "4leaf4site_train", "harrington-small_0_to_30_sites_subset", "harrington-small_0_to_50_taxa_subset", "harrington_tiny_test", "tiny_test"]
all_tree_list = []
for file in glob.glob("*.p"):
    file = os.path.realpath(file)
    if "2024" in file:
        continue
    with open(file, "rb") as f:
        data = pickle.load(f)
    data_name = file.split("/")[-1][:-2]
    if data_name not in restricted_files:
        continue
    print(data_name)
    this_data_list = []
    for tree in data:
        num_mutations = [[data_name, hamming_distance(node.sequence, node.up.sequence)] for node in tree.traverse() if node.up != None and not node.is_leaf()]
        this_data_list += num_mutations
    all_tree_list += this_data_list

30leaf_perfect_distinct_trees_train
harrington_tiny_test
10leaf_perfect_distinct_trees_train
4leaf4site_train
harrington-small_0_to_50_taxa_subset
tiny_test
harrington-small_0_to_30_sites_subset


In [4]:
all_tree_list

[['4leaf4site_test', 0],
 ['4leaf4site_test', 2],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 2],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 1],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 1],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 2],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 1],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 2],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 1],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 2],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 2],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 1],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 1],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 2],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 1],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 1],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 2],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 1],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 1],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 2],
 ['4leaf4site_test', 0],
 ['4leaf4site_test', 2],


In [29]:
# Create a Pandas DataFrame from the list of dictionaries
mut_per_edge_df = pd.DataFrame(all_tree_list, columns=["data_name", "num_mut"])
mut_per_edge_df = mut_per_edge_df.sort_values("data_name")
print(mut_per_edge_df)
plt.clf()
plt.figure(figsize=(10, 10))
# sns.barplot(data=mut_per_edge_df, x="num_mut", y="data_name", ci=None, 
# estimator='mean')
# sns.violinplot(data=mut_per_edge_df, x="num_mut", y="data_name")
# sns.histplot(data=mut_per_edge_df, x="num_mut", hue="data_name")
# Aggregate the data to count occurrences of num_mut for each data_name
count_df = mut_per_edge_df.groupby(['data_name', 'num_mut']).size().reset_index(name='count')

# Calculate total counts for each data_name
total_counts = count_df.groupby('data_name')['count'].sum().reset_index(name='total_count')

# Merge total counts back into the count_df
count_df = count_df.merge(total_counts, on='data_name')

# Calculate the fraction
count_df['fraction'] = count_df['count'] / count_df['total_count']

# Create the line plot
sns.lineplot(data=count_df, x='num_mut', y='fraction', hue='data_name')
plt.xscale("symlog")
plt.tight_layout()
plt.savefig("num_mutations_per_int_edge_lineplot.pdf")
plt.clf()

# alternative: histogram
sns.barplot(data=count_df, x='num_mut', y='fraction', hue='data_name')
plt.xlim(right=3.5, left=-0.5)
plt.tight_layout()
plt.savefig("num_mutations_per_int_edge_histogram.pdf")
plt.clf()

                                  data_name  num_mut
36265   10leaf_perfect_distinct_trees_train        1
37075   10leaf_perfect_distinct_trees_train        1
37076   10leaf_perfect_distinct_trees_train        1
37077   10leaf_perfect_distinct_trees_train        1
37078   10leaf_perfect_distinct_trees_train        2
...                                     ...      ...
109013                            tiny_test        0
109015                            tiny_test        0
109016                            tiny_test        2
109005                            tiny_test        0
109014                            tiny_test        2

[121929 rows x 2 columns]


/home/lcollien/.local/lib/python3.9/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/home/lcollien/.local/lib/python3.9/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/home/lcollien/.local/lib/python3.9/site-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/home/lcollien/.local/lib/python3.9/site-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, yo

# Number of trees and number of node pairs in training/validation sets

In [2]:
import pickle

filenames = ["4leaf4site_test.p", "4leaf4site_train.p", "10leaf_perfect_distinct_trees_test.p", "10leaf_perfect_distinct_trees_train.p", "30leaf_perfect_distinct_trees_test.p", "30leaf_perfect_distinct_trees_train.p", "harrington-small_0_to_30_sites_subset.p", "harrington-small_0_to_50_taxa_subset.p", "harrington_tiny_test.p", "harrington_very_tiny_test.p", "harrington-small_2024-06-10_train.p", "harrington-small_2024-06-10_test.p"]

for filename in filenames:
	with open(filename, "rb") as f:
		data = pickle.load(f)
	print(filename)
	print("Number of trees:", len(data))
	num_tree_pairs = 0
	min_seq_length = 1000
	max_seq_length = 0
	for tree in data:
		num_tree_pairs += 2 * (len(tree) - 1)
		min_seq_length = min(min_seq_length, len(tree.sequence))
		max_seq_length = max(max_seq_length, len(tree.sequence))
	print("Number of node pairs for training:", num_tree_pairs)
	print("Minimum sequence length:", min_seq_length)
	print("Maximum sequence length:", max_seq_length)

4leaf4site_test.p
Number of trees: 48
Number of node pairs for training: 192
Minimum sequence length: 4
Maximum sequence length: 4
4leaf4site_train.p
Number of trees: 192
Number of node pairs for training: 768
Minimum sequence length: 4
Maximum sequence length: 4
10leaf_perfect_distinct_trees_test.p
Number of trees: 205
Number of node pairs for training: 3690
Minimum sequence length: 3
Maximum sequence length: 8
10leaf_perfect_distinct_trees_train.p
Number of trees: 819
Number of node pairs for training: 14742
Minimum sequence length: 3
Maximum sequence length: 8
30leaf_perfect_distinct_trees_test.p
Number of trees: 205
Number of node pairs for training: 11890
Minimum sequence length: 15
Maximum sequence length: 24
30leaf_perfect_distinct_trees_train.p
Number of trees: 819
Number of node pairs for training: 47502
Minimum sequence length: 15
Maximum sequence length: 24
harrington-small_0_to_30_sites_subset.p
Number of trees: 340
Number of node pairs for training: 25824
Minimum sequence 

# Number of sites

In [4]:
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

In [10]:
filenames = ["4leaf4site_test.p", "4leaf4site_train.p", "10leaf_perfect_distinct_trees_test.p", "10leaf_perfect_distinct_trees_train.p", "30leaf_perfect_distinct_trees_test.p", "30leaf_perfect_distinct_trees_train.p", "harrington-small_0_to_30_sites_subset.p", "harrington-small_0_to_50_taxa_subset.p", "harrington_tiny_test.p", "harrington_very_tiny_test.p", "harrington-small_2024-06-10_train.p", "harrington-small_2024-06-10_test.p"]

seq_lengths = []
for filename in filenames:
	with open(filename, "rb") as f:
		data = pickle.load(f)
	dataname = filename[:-1]
	print(filename)
	print("Number of trees:", len(data))
	for tree in data:
		seq_lengths.append([dataname, len(tree.sequence)])

4leaf4site_test.p
Number of trees: 48
4leaf4site_train.p
Number of trees: 192
10leaf_perfect_distinct_trees_test.p
Number of trees: 205
10leaf_perfect_distinct_trees_train.p
Number of trees: 819
30leaf_perfect_distinct_trees_test.p
Number of trees: 205
30leaf_perfect_distinct_trees_train.p
Number of trees: 819
harrington-small_0_to_30_sites_subset.p
Number of trees: 340
harrington-small_0_to_50_taxa_subset.p
Number of trees: 1711
harrington_tiny_test.p
Number of trees: 100
harrington_very_tiny_test.p
Number of trees: 30
harrington-small_2024-06-10_train.p
Number of trees: 25039
harrington-small_2024-06-10_test.p
Number of trees: 6260


In [14]:
df = pd.DataFrame(seq_lengths, columns = ["dataname", "num_sites"])
print(df)

# Group by 'dataname' and calculate mean of 'num_sites'
mean_num_sites = df.groupby('dataname')['num_sites'].mean().reset_index()
# Rename the column for clarity
mean_num_sites.rename(columns={'num_sites': 'mean_num_sites'}, inplace=True)
print(mean_num_sites)

plt.clf()
plt.figure(figsize=(10, 10))
sns.boxplot(data=df, x="num_sites", y="dataname")
plt.tight_layout()
plt.savefig("alignment_lengths.pdf")
plt.clf()

                                dataname  num_sites
0                       4leaf4site_test.          4
1                       4leaf4site_test.          4
2                       4leaf4site_test.          4
3                       4leaf4site_test.          4
4                       4leaf4site_test.          4
...                                  ...        ...
35763  harrington-small_2024-06-10_test.        129
35764  harrington-small_2024-06-10_test.         57
35765  harrington-small_2024-06-10_test.         75
35766  harrington-small_2024-06-10_test.        129
35767  harrington-small_2024-06-10_test.        129

[35768 rows x 2 columns]
                                  dataname  mean_num_sites
0      10leaf_perfect_distinct_trees_test.        5.756098
1     10leaf_perfect_distinct_trees_train.        5.799756
2      30leaf_perfect_distinct_trees_test.       19.458537
3     30leaf_perfect_distinct_trees_train.       19.356532
4                         4leaf4site_test.        4.000